### **CIFAR-10 dataset**

In [ ]:
import tensorflow as tf
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Load the CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train, y_train = x_train[:5000]/255., y_train[:5000]
x_train = np.transpose(x_train, (0, 3, 1, 2))
x_test, y_test = x_test[:1000]/255., y_test[:1000]
x_test = np.transpose(x_test, (0, 3, 1, 2))

print(x_train.shape, x_train.dtype)
print(y_train.shape, y_train.dtype)
print(x_test.shape, x_test.dtype)
print(y_test.shape, y_test.dtype)

(5000, 3, 32, 32) float64
(5000, 1) uint8
(1000, 3, 32, 32) float64
(1000, 1) uint8


In [ ]:
class CIFAR(torch.utils.data.Dataset):
    def __init__(self, x, y, mode):
        self.x = x.astype(np.float32)
        self.y = y.astype(np.int64)
        self.mode = mode

        self.t_train = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.t_not_train = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.x[idx])
        if self.mode == 'train':
          img = self.t_train(img)
        else:
          img = self.t_not_train(img)

        return img, self.y[idx]

train_dataset = CIFAR(x_train[:4000], y_train[:4000], 'train')
val_dataset = CIFAR(x_train[4000:], y_train[4000:], 'val')
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=100, num_workers=4, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=100, shuffle=False)

In [ ]:
# compute accuracy
def acc(train_dataloader, model):
  total = 0
  correct = 0
  for x, y in train_dataloader:
    with torch.no_grad():
      y_ = model(x.cuda()).detach().cpu()
    y_ = torch.argmax(y_, dim=1)

    #print(y_.size(), y[:,0].size(), (y_ == y[:,0]).size())

    correct += torch.sum((y_ == y[:,0]).float())
    total += len(x)

    #break
  return (correct/total).item()

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(400, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net().cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=100)

best_acc = 0.0
for i in range(100):
  for x, y in train_dataloader:
    # zero the parameter gradients
    optimizer.zero_grad()

    # forward + backward + optimize
    outputs = net(x.cuda())
    loss = criterion(outputs, y[:,0].cuda())
    loss.backward()
    optimizer.step()

  val_acc = acc(val_dataloader, net)
  print(i, loss.item(), acc(train_dataloader, net), val_acc)

  if val_acc > best_acc:
    print('saved at', i)
    torch.save(net.state_dict(), 'mymodel.pth')
    best_acc = val_acc

  scheduler.step()

print('Finished Training')

0 2.0231027603149414 0.25874999165534973 0.24500000476837158
saved at 0
1 1.754992961883545 0.335750013589859 0.32499998807907104
saved at 1
2 1.6823620796203613 0.3569999933242798 0.35100001096725464
saved at 2
3 1.7793338298797607 0.39750000834465027 0.39800000190734863
saved at 3
4 1.6574912071228027 0.40825000405311584 0.40400001406669617
saved at 4
5 1.5756927728652954 0.4269999861717224 0.4059999883174896
saved at 5
6 1.501743197441101 0.44600000977516174 0.41499999165534973
saved at 6
7 1.533167839050293 0.4657500088214874 0.4390000104904175
saved at 7
8 1.4610134363174438 0.4632500112056732 0.42800000309944153
9 1.5545470714569092 0.4807499945163727 0.42399999499320984
10 1.2971805334091187 0.5007500052452087 0.43299999833106995
11 1.5952234268188477 0.49000000953674316 0.44699999690055847
saved at 11
12 1.4687100648880005 0.512499988079071 0.46700000762939453
saved at 12
13 1.40468430519104 0.5249999761581421 0.4410000145435333
14 1.4027581214904785 0.5322499871253967 0.460999

In [ ]:
net.load_state_dict(torch.load('mymodel.pth'))

test_dataset = CIFAR(x_test, y_test, 'test')
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False)

print(acc(test_dataloader, net))

0.5199999809265137


In [ ]:
# compute accuracy
def acc(train_dataloader, model):
  model.eval()

  total = 0
  correct = 0
  for x, y in train_dataloader:
    with torch.no_grad():
      y_ = model(x.cuda()).detach().cpu()
    y_ = torch.argmax(y_, dim=1)

    correct += torch.sum((y_ == y[:,0]).float())
    total += len(x)

  return (correct/total).item()

model = torchvision.models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(512, 10)
model = model.cuda()

#for name, param in model.named_parameters():
#  print(name, param.shape)

optimizer1 = optim.Adam([p for n,p in model.named_parameters() if n.startswith('fc.')], lr=0.001)
optimizer2 = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

best_acc = 0.0
for i in range(100):
  if i < 10:
    optimizer = optimizer1
  else:
    optimizer = optimizer2

  model.train()
  for x, y in train_dataloader:
    # zero the parameter gradients
    optimizer.zero_grad()

    # forward + backward + optimize
    outputs = model(x.cuda())
    loss = criterion(outputs, y[:,0].cuda())
    loss.backward()
    optimizer.step()

  val_acc = acc(val_dataloader, model)
  print(i, loss.item(), acc(train_dataloader, model), val_acc)

  if val_acc > best_acc:
    print('saved at', i)
    torch.save(model.state_dict(), 'mymodel.pth')
    best_acc = val_acc

print('Finished Training')

0 2.131420612335205 0.3017500042915344 0.31200000643730164
saved at 0
1 1.9252617359161377 0.37549999356269836 0.36800000071525574
saved at 1
2 1.6707885265350342 0.3957499861717224 0.3700000047683716
saved at 2
3 1.7471646070480347 0.4169999957084656 0.3799999952316284
saved at 3
4 1.5991134643554688 0.42625001072883606 0.3959999978542328
saved at 4
5 1.7783945798873901 0.4449999928474426 0.4020000100135803
saved at 5
6 1.6063103675842285 0.4555000066757202 0.39100000262260437
7 1.8089178800582886 0.4544999897480011 0.39800000190734863
8 1.7361211776733398 0.45750001072883606 0.4129999876022339
saved at 8
9 1.7140188217163086 0.4625000059604645 0.40299999713897705
10 1.1784778833389282 0.6269999742507935 0.5759999752044678
saved at 10
11 0.9115670919418335 0.7242500185966492 0.6520000100135803
saved at 11
12 0.8897372484207153 0.7492499947547913 0.6460000276565552
13 0.9040448069572449 0.7975000143051147 0.6759999990463257
saved at 13
14 0.5861449241638184 0.8372499942779541 0.6959999

In [ ]:
# how to add a normalization transformation to a traced model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(400, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        self.t = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def forward(self, x):
        x = self.t(x)

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x